# Prepare regions for upload to MorphoSource

We looked into uploading datasets for sharing, see https://github.com/habi/sticklebacks-manuscript/issues/11
We figured out, that [MorphoSource](https://www.morphosource.org/) is probably the best thing to try.
This notebook is used to prepare the `rec_regions` exports written by the `BucketSeparator.ipynb` notebook for upload to there.

The cells below are used to set up the whole notebook.
They load needed libraries and set some default values.

In [1]:
# Load the modules we need
import platform
import os
import glob
import pandas
import imageio
import numpy
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn
import dask
import dask_image.imread
from dask.distributed import Client, LocalCluster
from tqdm.auto import tqdm, trange

In [2]:
# Load our own log file parsing code
# This is loaded as a submodule to alleviate excessive copy-pasting between *all* projects we do
# See https://github.com/habi/BrukerSkyScanLogfileRuminator for details on its inner workings
import BrukerSkyScanLogfileRuminator.parsing_functions as logparse

In [3]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
# We use the fast internal SSD for speed reasons
import tempfile
if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

Dask temporary files go to /media/habi/Fast_SSD/tmp


In [4]:
from dask.distributed import Client
client = Client()

In [5]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
# plt.rcParams['figure.figsize'] = (16 * 0.618, 9 * 0.618)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 300

In [6]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

In [7]:
# Display all plots identically
lines = 3
# And then do something like
# plt.subplot(lines, int(numpy.ceil(len(Data) / float(lines))), c + 1)

Since the (tomographic) data can reside on different drives we set a folder to use below

In [8]:
# Different locations if running either on Linux or Windows
FastSSD = True
if 'Linux' in platform.system():
    if FastSSD:
        BasePath = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
    else:
        BasePath = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
elif 'Windows' in platform.system():
    if FastSSD:
        BasePath = os.path.join('F:\\')
    else:
        BasePath = os.path.join('N:\\')
if 'research_storage_ben' in BasePath:
    Root = os.path.join(BasePath)
else:
    Root = os.path.join(BasePath, 'IEE Stickleback')
# Force reading from Bens research storage folder
# Root = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
print('We are loading all the data from %s' % Root)

We are loading all the data from /media/habi/Fast_SSD/IEE Stickleback


Now that we are set up, actually start to load/ingest the data.

In [9]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [10]:
# Get *all* log files present on disk
# Using os.walk is way faster than using recursive glob.glob
# Not sorting the found logfiles is also making it quicker
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

In [11]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]

In [12]:
# Show a (small) sampler of the loaded data as a first check
Data.sample(n=5)

,LogFile,Folder
659,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
458,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
35,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
644,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
666,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...


In [13]:
# Check for samples which are not yet reconstructed
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row.Folder:
        if 'TScopy' not in row.Folder and 'PR' not in row.Folder:
            # If there's nothing with 'rec*' on the same level, then tell us
            if not glob.glob(row.Folder.replace('proj', '*rec*')):
                print('- %s is missing matching reconstructions' % row.LogFile[len(Root) + 1:])
# Sticklebucket_14/proj2/Sticklebucket_14~00.log and 
# Sticklebucket_15/proj2/Sticklebucket_15~00.log are failed scans where we cannot do a reconstruction

- Sticklebucket_15/proj2/Sticklebucket_15~00.log is missing matching reconstructions
- Sticklebucket_14/proj2/Sticklebucket_14~00.log is missing matching reconstructions
- BucketOfFish_C/proj/Sticklebucket_C.log is missing matching reconstructions
- BucketOfFish_C/proj/Sticklebucket_C~01.log is missing matching reconstructions
- BucketOfFish_C/proj/Sticklebucket_C~00.log is missing matching reconstructions
- BucketOfFish_C/proj/Sticklebucket_C~02.log is missing matching reconstructions


In [14]:
# Search for any .csv files in each folder.
# These are only generated when the "X/Y Alignment With a Reference Scan" was performed in NRecon.
# If those files do *not* exist we have missed to do it and should correct for this.
Data['XYAlignment'] = [glob.glob(os.path.join(f, '*T*.csv')) for f in Data['Folder']]

In [15]:
# Display samples which are missing the .csv-files for the XY-alignment
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row['Folder']:
        if not row['XYAlignment']:
            if not any(x in row.LogFile for x in ['rectmp.log',  # because we only exclude temporary logfiles in a later step
                                                  'proj_nofilter',  # since these two scans of single teeth don't contain a reference scan
                                                  'TScopy',  # discard *t*hermal *s*hift data
                                                  ]):
                print('- %s has *not* been X/Y aligned' % row.LogFile[len(Root) + 1:])

- Sticklebucket_15/proj/Sticklebucket_15~00.log has *not* been X/Y aligned
- Sticklebucket_15/proj2/Sticklebucket_15~00.log has *not* been X/Y aligned
- Sticklebucket_14/proj3/Sticklebucket_14~02.log has *not* been X/Y aligned
- Sticklebucket_14/proj3/Sticklebucket_14.log has *not* been X/Y aligned
- Sticklebucket_14/proj3/Sticklebucket_14~00.log has *not* been X/Y aligned
- Sticklebucket_14/proj3/Sticklebucket_14~01.log has *not* been X/Y aligned
- Sticklebucket_14/proj2/Sticklebucket_14~00.log has *not* been X/Y aligned


In [16]:
# Get rid of all the logfiles from all the folders that might be on disk but that we don't want to load the data from
for c, row in Data.iterrows():
    if os.path.split(row.Folder)[-1] == 'proj':  # drop all projections folders
        Data.drop([c], inplace=True)
    elif 'ucket' not in row.Folder:  # Remove all test scans which are not named 'Sticklbucket_*' or something else containing 'ucket'
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Exclude all log files that we write in ourselves
        Data.drop([c], inplace=True)
    elif os.path.split(row.LogFile)[1].startswith('._'):  # Remove macos metadata files for files on external storage
        Data.drop([c], inplace=True)        
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

It's a bit silly to exclude the self-written log files (`*_regions/*/*.log`) above, but like so we know for sure to include everything we've exported...

In [17]:
# Generate us some meaningful colums in the dataframe
Data['Sample'] = [os.path.basename(log).replace('_rec.log', '') for log in Data['LogFile']]
Data['Scan'] = [os.path.basename(os.path.dirname(log)) for log in Data['LogFile']]

In [18]:
# Show the data from the last loaded scans
Data.tail(n=5)

,LogFile,Folder,XYAlignment,Sample,Scan
65,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,[],Sticklebucket_U,rec
66,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,[],Sticklebucket_D,rec
67,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,[],Sticklebucket_B,rec
68,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,/media/habi/Fast_SSD/IEE Stickleback/BucketOfF...,[],Sticklebucket_M,rec
69,/media/habi/Fast_SSD/IEE Stickleback/Sticklebu...,/media/habi/Fast_SSD/IEE Stickleback/Sticklebu...,[],Sticklebucket_08,rec


In [19]:
# Load the file names of all the reconstructions of all the scans
Data['Filenames Reconstructions'] = [sorted(glob.glob(os.path.join(f, '*rec0*.png'))) for f in Data['Folder']]
# How many reconstructions do we have?
Data['Number of reconstructions'] = [len(r) for r in Data['Filenames Reconstructions']]

In [20]:
# Drop samples which have either not been reconstructed yet or of which we deleted the reconstructions with
# `find . -name "*rec*.png" -type f -mtime +333 -delete`
# Based on https://stackoverflow.com/a/13851602
# for c,row in Data.iterrows():
#     if not row['Number of reconstructions']:
#         print('%s contains no PNG files, we might be currently reconstructing it' % row.Folder)
Data = Data[Data['Number of reconstructions'] > 0]
# Reset the dataframe count/index for easier indexing afterwards
Data.reset_index(drop=True, inplace=True)
print('We have %s folders with reconstructions' % (len(Data)))

We have 23 folders with reconstructions


In [21]:
# Get parameters to doublecheck from logfiles
Data['Voxelsize'] = [logparse.pixelsize(log) for log in Data['LogFile']]
Data['Filter'] = [logparse.whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [logparse.exposuretime(log) for log in Data['LogFile']]
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['Averaging'] = [logparse.averaging(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [logparse.projection_size(log) for log in Data['LogFile']]
Data['RotationStep'] = [logparse.rotationstep(log) for log in Data['LogFile']]
Data['Grayvalue'] = [logparse.reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [logparse.ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [logparse.beamhardening(log) for log in Data['LogFile']]
Data['DefectPixelMasking'] = [logparse.defectpixelmasking(log) for log in Data['LogFile']]
Data['Scan date'] = [logparse.scandate(log) for log in Data['LogFile']]

In [22]:
# Sort dataframe based on the scan date
Data.sort_values(by=['Scan date'],
                 ignore_index=True,
                 inplace=True)

Now we 'load' all reconstructions from disks into stacks.

In [23]:
# Load all reconstructions into ephemereal DASK arrays, with a nice progress bar...
Reconstructions = [None] * len(Data)
for c, row in tqdm(Data.iterrows(),
                   desc='Loading reconstructions',
                   total=len(Data)):
    Reconstructions[c] = dask_image.imread.imread(os.path.join(row['Folder'], '*rec*.png'))[:, :, :, 0]  # Get rid of the color channel

Loading reconstructions:   0%|          | 0/23 [00:00<?, ?it/s]

/home/habi/miniconda3/envs/sticklebacks/lib/python3.12/site-packages/pims/image_sequence.py:85: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  return imread(filename, **kwargs)
/home/habi/miniconda3/envs/sticklebacks/lib/python3.12/site-packages/pims/image_sequence.py:85: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  return imread(filename, **kwargs)
/home/habi/miniconda3/envs/sticklebacks/lib/python3.12/site-packages/pims/image_sequence.py:85: FutureWarning: The plugin infrastructure in `s

In [24]:
Reconstructions[0]

dask.array<getitem, shape=(4094, 1536, 1536), dtype=uint8, chunksize=(1, 1536, 1536), chunktype=numpy.ndarray>

In [25]:
# What do we have on disk?
print('We have %s reconstructions on %s' % (Data['Number of reconstructions'].sum(), Root))
print('This is about %s reconstructions per scan (%s scans in %s folders)' % (round(Data['Number of reconstructions'].sum() / len(Data)),
                                                                              len(Data),
                                                                              len(Data.Sample.unique())))

We have 96577 reconstructions on /media/habi/Fast_SSD/IEE Stickleback
This is about 4199 reconstructions per scan (23 scans in 20 folders)


In [26]:
# How big are the datasets?
Data['Size'] = [rec.shape for rec in Reconstructions]

In [27]:
# The three cardinal directions
directions = ['Axial',
              'Frontal',
              'Median']

In [28]:
# Read or calculate the directional MIPs, put them into the dataframe and save them to disk
for d, direction in enumerate(directions):
    Data['MIP_' + direction] = ''
for c, row in tqdm(Data.iterrows(), desc='Working on MIPs', total=len(Data)):
    for d, direction in tqdm(enumerate(directions),
                             desc='%s/%s' % (row['Sample'], row['Scan']),
                             leave=False,
                             total=len(directions)):
        outfilepath = os.path.join(os.path.dirname(row['Folder']),
                                   '%s.%s.MIP.%s.png' % (row['Sample'], row['Scan'], direction))
        if not os.path.exists(outfilepath):
            # Generate and save MIP
            imageio.imwrite(outfilepath, Reconstructions[c].max(axis=d).compute().astype('uint8'))
        Data.at[c, 'MIP_' + direction] = dask_image.imread.imread(outfilepath).squeeze()

Working on MIPs:   0%|          | 0/23 [00:00<?, ?it/s]

Sticklebucket/18um_rec:   0%|          | 0/3 [00:00<?, ?it/s]

Sticklebucket/15um_rec:   0%|          | 0/3 [00:00<?, ?it/s]

Sticklebucket/17.5um_rec:   0%|          | 0/3 [00:00<?, ?it/s]

Sticklebucket/19um_rec:   0%|          | 0/3 [00:00<?, ?it/s]

...

StickleBucket_Q/rec:   0%|          | 0/3 [00:00<?, ?it/s]

StickleBucket_R/rec:   0%|          | 0/3 [00:00<?, ?it/s]

Sticklbucket_S/rec:   0%|          | 0/3 [00:00<?, ?it/s]

Sticklebucket_T/rec:   0%|          | 0/3 [00:00<?, ?it/s]

Sticklebucket_U/rec:   0%|          | 0/3 [00:00<?, ?it/s]

Since we've done everything *correctly* in `BucketSeparator.ipynb` we can simply go through all the desired folders and pull all in from disk.
This is more efficient than re-doing the extraction from scratch :)

In [29]:
# Construct folder name for regions folder
Data['FolderRegionsExports'] = None
for c, row in Data.iterrows():
    Data.at[c, 'FolderRegionsExports'] = os.path.join(os.path.dirname(os.path.dirname(row.LogFile)), row.Scan + '_regions')

In [30]:
# Search for log files we've written and construct the regions names from theseFind folders in each regions folder and put them into the dataframe
Data['RegionsLogFiles'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsLogFiles'] = sorted(glob.glob(os.path.join(row.FolderRegionsExports, '*', '*.log')))    

In [31]:
Data.RegionsLogFiles[3]

['/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.018/FG.X23.018.log',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.019/FG.X23.019.log',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.020/FG.X23.020.log',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.021/FG.X23.021.log',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.022/FG.X23.022.log',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.023/FG.X23.023.log']

In [32]:
# Construct regions name (and double-check for errors on the way)Search for log files we've written and construct the regions names from theseFind folders in each regions folder and put them into the dataframe
Data['RegionsName'] = None
Data['RegionsFolder'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsName'] = [os.path.splitext(os.path.basename(logfilename))[0] for logfilename in row.RegionsLogFiles]
    Data.at[c, 'RegionsFolder'] = [os.path.dirname(logfilename) for logfilename in row.RegionsLogFiles]
    for rn, rf in zip(Data.at[c, 'RegionsName'], Data.at[c, 'RegionsFolder']):
        if rn != os.path.basename(rf):  # Tested with a manual rename on disk :)
            print(f'Error: For {row.LogFile[len(Root):]}: Extracted Region name "{rn}" does not match extracted folder name "{os.path.basename(rf)}"')

In [33]:
Data.RegionsName[3]

['FG.X23.018',
 'FG.X23.019',
 'FG.X23.020',
 'FG.X23.021',
 'FG.X23.022',
 'FG.X23.023']

In [34]:
Data.RegionsFolder[3]

['/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.018',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.019',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.020',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.021',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.022',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.023']

In [35]:
Data.LogFile

0     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
1     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
2     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
3     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
4     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
5     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
6     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
7     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
8     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
9     /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
10    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
11    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
12    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
13    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
14    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
15    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
16    /media/habi/Fast_SSD/IEE Stickleback/BucketOfF...
17    /media/habi/Fast_SSD/IEE Stickleback/Bucke

MorphoSource would like to ingest a ".zip containing .tif, .jpeg, .bmp, or .dcm*", see https://docs.google.com/document/d/1QByWl5t0SFD4QkdxUdoUbeTNms3HEhQYdTndo6PR-Ts/edit?tab=t.0, so we're preparing these files.
In [a test](https://www.morphosource.org/concern/media/000885110?locale=en), we've seen that a .zip with PNGs works fine, too...

Each .zip file should contain the original `proj/*.log`, `rec/*.log` and all the files from `rec_regions/FishID/*` for reproducible research.

In [36]:
Data.RegionsFolder[3]

['/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.018',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.019',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.020',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.021',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.022',
 '/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/19um_rec_regions/FG.X23.023']

In [77]:
Data.Folder[0]

'/media/habi/Fast_SSD/IEE Stickleback/BucketOfFish_A/18um_rec'

In [102]:
# Define us a "custom" zipping function
import pathlib
import zipfile

def zip_folder(folder, logfile, scan_date):
    # Generate folder names
    folder = pathlib.Path(folder)
    logfile = pathlib.Path(logfile)

    # Generate path for the zip file
    zip_path = folder.parent / (folder.name + ".zip")

    # Search for correct label-checking file
    search_string = folder.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        folder.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )

    # Create README file (with function below)
    readme_path = create_readme(folder, logfile, scan_date)

    # Actually do the zipping now
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        # Add region folder
        for file in folder.rglob("*"):
            zf.write(file, arcname=file.relative_to(folder.parent))

        # Add reconstruction log file
        zf.write(logfile, arcname=logfile.name)

        # Add label-checking file
        zf.write(labelcheckingfile, arcname=labelcheckingfile.name)

        # Add README
        zf.write(readme_path, arcname="README.md")

    # Remove temporary README
    readme_path.unlink()

    return zip_path

In [ ]:
def get_file_listing(region):
    from pathlib import Path

    region = Path(region)

    files = sorted([f.relative_to(region) for f in region.rglob("*") if f.is_file()])

    pngs = [f for f in files if f.suffix.lower() == ".png"]
    regionlog = [f for f in files if f.suffix.lower() == ".log"]

    return pngs, regionlog

In [ ]:
# We want to add a custom/dynamic README.md file to *every* archive, so let's generate one
from datetime import datetime

def create_readme(region, logfile, scan_date):
    region = pathlib.Path(region)
    logfile = pathlib.Path(logfile)
    pngs, regionlog = get_file_listing(region)
    print(pngs)
    print(regionlog)
    if len(pngs) > 2:
        png_listing = (
            f"│   ├── {pngs[0]}\n"
            f"│   ├── {pngs[1]}\n"
            f"│   ├── ...\n"
            f"│   └── {pngs[-1]}"
        )
    else:
        png_listing = "\n".join(f"│   └── {p}" for p in pngs)

    log_listing = "\n".join(f"│   └── {l}" for l in regionlog)

    readme = f"""# {region.name}

This archive was generated on {datetime.now().isoformat(timespec="seconds")} with https://github.com/habi/sticklebacks-manuscript, as part of https://habi.github.io/sticklebacks-manuscript/
It contains the cropped reconstructions for specimen: **{region.name}**, which was scanned on {scan_date}.

## Archive contents

{region.name}.zip
├── {region.name}/
│   ├── PNG slices ({len(pngs)} files)
{png_listing}
{log_listing}
├── {logfile.name}`: log file of the reconstructions of the original multi-specimen scan.
└── README.md: This file

## Source

Extracted region folder: {region.relative_to(Root)}
Log file of the reconstructions of the original multi-specimen scan: {logfile.relative_to(Root)}

"""

    readme_path = region.parent / "README.md"
    readme_path.write_text(readme, encoding="utf-8")

    return readme_path

In [105]:
for c, row in tqdm(Data.iterrows(), desc='Zipping', total=len(Data)):
    for d, region in tqdm(enumerate(row.RegionsFolder),
                          desc=f'Zipping regions of {os.path.dirname(row.LogFile[len(Root):])}',
                          total=len(row.RegionsFolder),
                          leave=False):
        zip_folder(region, row.LogFile, row['Scan date'])
        

Zipping:   0%|          | 0/23 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_A/18um_rec:   0%|          | 0/4 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_A/15um_rec:   0%|          | 0/6 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_A/17.5um_rec:   0%|          | 0/12 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_A/19um_rec:   0%|          | 0/6 [00:00<?, ?it/s]

...

Zipping regions of /BucketOfFish_Q/rec:   0%|          | 0/12 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_R/rec:   0%|          | 0/12 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_S/rec:   0%|          | 0/12 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_T/rec:   0%|          | 0/12 [00:00<?, ?it/s]

Zipping regions of /BucketOfFish_U/rec:   0%|          | 0/8 [00:00<?, ?it/s]